# Example 02 — Olivetti Faces (High-Dimensional)

This notebook reproduces the benchmark from the original project notebook,
showing FuzzyARTMAP outperforming XGBoost on the Olivetti faces dataset
(400 samples, 4096 features, 40 classes).

**Key concepts covered:**
- `TruncatedSVD` dimensionality reduction before ARTMAP
- `partial_fit` for online / streaming learning
- Benchmarking against XGBoost

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from sklearn.datasets import fetch_olivetti_faces
from sklearn.metrics import classification_report, f1_score
from sklearn.model_selection import train_test_split

from fuzzyart import FuzzyARTMAP
from fuzzyart.preprocessing import normalize

## 1. Load Olivetti faces

In [ ]:
data = fetch_olivetti_faces()
X, y = data.data, data.target
print(f'Shape: {X.shape}  Classes: {len(np.unique(y))}')

# The dataset is already in [0, 1] (float32), but normalise to be safe
X = normalize(X.astype(np.float64))

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=0
)

## 2. Optional: reduce dimensionality

4096 features with complement coding → 8192 dimensions.  
Truncated SVD reduces this while preserving class structure.

In [ ]:
# Full resolution (4096 features) — slower but more accurate
# For a faster run, uncomment SVD reduction:
# X_train_r = normalize(truncated_svd(X_train, n_components=100))
# X_test_r  = normalize(truncated_svd(X_test,  n_components=100))

X_train_r, X_test_r = X_train, X_test
print(f'Feature dim after reduction: {X_train_r.shape[1]}')

## 3. Train FuzzyARTMAP

In [ ]:
clf = FuzzyARTMAP(alpha=0.3, beta=0.1, rho_baseline=0.0, epochs=4, verbose=True)
clf.fit(X_train_r, y_train)
print(f'Committed nodes: {clf.n_committed_}')

## 4. Evaluate FuzzyARTMAP

In [ ]:
y_pred_fam = clf.predict(X_test_r)
fam_f1 = f1_score(y_test, y_pred_fam, average='weighted')
print(f'FuzzyARTMAP weighted F1: {fam_f1:.3f}')
print(classification_report(y_test, y_pred_fam))

## 5. Benchmark vs XGBoost

In [ ]:
try:
    from xgboost import XGBClassifier
    xgb = XGBClassifier(n_estimators=100, use_label_encoder=False, eval_metric='mlogloss')
    xgb.fit(X_train_r, y_train)
    y_pred_xgb = xgb.predict(X_test_r)
    xgb_f1 = f1_score(y_test, y_pred_xgb, average='weighted')
    print(f'XGBoost   weighted F1: {xgb_f1:.3f}')

    fig, ax = plt.subplots(figsize=(5, 3))
    ax.bar(['FuzzyARTMAP', 'XGBoost'], [fam_f1, xgb_f1], color=['#2196F3', '#FF5722'])
    ax.set_ylabel('Weighted F1'); ax.set_title('Olivetti Faces — Model Comparison')
    ax.set_ylim(0, 1.05)
    plt.tight_layout(); plt.show()
except ImportError:
    print('xgboost not installed — skipping comparison')

## 6. Online learning with partial_fit

In [ ]:
# Simulate streaming: train one sample at a time
online_clf = FuzzyARTMAP(alpha=0.3, beta=0.1)
f1_history = []

for i in range(len(X_train_r)):
    online_clf.partial_fit(X_train_r[i:i+1], y_train[i:i+1])
    if (i + 1) % 20 == 0:
        preds = online_clf.predict(X_test_r)
        f1_history.append(f1_score(y_test, preds, average='weighted', zero_division=0))

plt.figure(figsize=(7, 3))
plt.plot(range(20, len(X_train_r)+1, 20), f1_history)
plt.xlabel('Training samples seen'); plt.ylabel('Test F1 (weighted)')
plt.title('Online Learning Curve — FuzzyARTMAP')
plt.tight_layout(); plt.show()